In [11]:
import numpy as np
import torch
import pickle
# Path to your .npy file
file_path = 'crypto_data1.npy'

# Load the .npy file
fea = np.load(file_path)
print(fea[1,0,:],fea.shape)
fea = fea[:,1:,1:].astype(float)
fea = torch.tensor(fea).transpose(0,1)
fea = torch.flip(fea,dims=(0,))
# Print the data·
print(fea.shape)
print(fea[:,0,-2])

['1722297600000' '3317.93' '3356.2' '3281.8' '3338.89' '0'] (181, 1440, 6)
torch.Size([1439, 181, 5])
tensor([11532.5000, 11673.7000, 11653.2000,  ..., 67909.2000, 68240.1000,
        66778.7000], dtype=torch.float64)


In [12]:
label = torch.ones_like(fea)

for i in range(fea.shape[1]):
    j = 0
    while fea[j,i,0] == 0 and j < fea.shape[0]:
        label[j,i,:] = 0
        j += 1        
    label[j:j+3,i,:] = 0
    fea[:,i,-1]=fea[:,i,-2]
    
    for k in range(4):
        tmpj = j 
        prev = fea[tmpj,i,k].item()
        fea[tmpj,i,k] = 0 
        tmpj += 1
        while tmpj<fea.shape[0]:
            # print(tmp,tmp2,fea[j,i,0],j)
            nxtv= fea[tmpj,i,k] / prev
            prev= fea[tmpj,i,k].item()
            fea[tmpj,i,k]= nxtv - 1
            tmpj += 1

print(fea.shape)
print(fea[0:10,0,0],'\n',fea[0:3,0,:])

torch.Size([1439, 181, 5])
tensor([ 0.0000, -0.0280,  0.0122, -0.0018,  0.0088, -0.0367,  0.0130, -0.0122,
         0.0179, -0.0050], dtype=torch.float64) 
 tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.1532e+04],
        [-2.7974e-02, -1.6482e-02, -9.5822e-03,  1.2244e-02,  1.1674e+04],
        [ 1.2244e-02,  2.4294e-03,  1.2926e-02, -1.7561e-03,  1.1653e+04]],
       dtype=torch.float64)


In [13]:
tmpl = 192
outl = 1

In [14]:
file_name = f'data_in{tmpl}_out{outl}.pkl'

data = {'processed_data':fea.numpy()}
with open(file_name, 'wb') as file:
    pickle.dump(data, file)

In [15]:
file_name = f'label_in{tmpl}_out{outl}.pkl'

data = {'processed_data':label.numpy()}
with open(file_name, 'wb') as file:
    pickle.dump(data, file)

In [16]:
# tmpl = 144
length = fea.shape[0]-tmpl-outl
testl = int(16)
validl = int(16)
trainl = int(length)-testl-validl
# trainl = 8
# testl = 8
# validl = 8
# print(trainl,testl,validl)
idx = {}
train = []
for i in range(0,trainl,1):
    train.append((i,i+tmpl,i+tmpl+outl))
idx['train']=train
valid=[]
for i in range(trainl,trainl+testl,1):
    valid.append((i,i+tmpl,i+tmpl+outl))
idx['valid']=valid
test=[]
for i in range(trainl+testl,trainl+testl+validl,1):
    test.append((i,i+tmpl,i+tmpl+outl))
idx['test'] =test

file_name = f'index_in{tmpl}_out{outl}.pkl'

with open(file_name,'wb') as file:
    pickle.dump(idx,file)

print(train)
print(len(idx['train']),len(idx['test']))

[(0, 192, 193), (1, 193, 194), (2, 194, 195), (3, 195, 196), (4, 196, 197), (5, 197, 198), (6, 198, 199), (7, 199, 200), (8, 200, 201), (9, 201, 202), (10, 202, 203), (11, 203, 204), (12, 204, 205), (13, 205, 206), (14, 206, 207), (15, 207, 208), (16, 208, 209), (17, 209, 210), (18, 210, 211), (19, 211, 212), (20, 212, 213), (21, 213, 214), (22, 214, 215), (23, 215, 216), (24, 216, 217), (25, 217, 218), (26, 218, 219), (27, 219, 220), (28, 220, 221), (29, 221, 222), (30, 222, 223), (31, 223, 224), (32, 224, 225), (33, 225, 226), (34, 226, 227), (35, 227, 228), (36, 228, 229), (37, 229, 230), (38, 230, 231), (39, 231, 232), (40, 232, 233), (41, 233, 234), (42, 234, 235), (43, 235, 236), (44, 236, 237), (45, 237, 238), (46, 238, 239), (47, 239, 240), (48, 240, 241), (49, 241, 242), (50, 242, 243), (51, 243, 244), (52, 244, 245), (53, 245, 246), (54, 246, 247), (55, 247, 248), (56, 248, 249), (57, 249, 250), (58, 250, 251), (59, 251, 252), (60, 252, 253), (61, 253, 254), (62, 254, 255), (

In [17]:
scaler = {}
scaler['func']='re_standard_transform'
scaler['args']={'mean': 0, 'std': 1}

file_name = f'scaler_in{tmpl}_out{outl}.pkl'

with open(file_name,'wb') as file:
    pickle.dump(scaler,file)

In [18]:
print(label[5,:,3])

tensor([1., 1., 1., 0., 0., 0., 1., 1., 1., 0., 1., 1., 0., 0., 1., 1., 0., 0.,
        1., 1., 0., 1., 0., 0., 1., 1., 0., 1., 0., 1., 1., 0., 1., 0., 1., 0.,
        0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0.,
        0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0.,
        0.], dtype=torch.float64)


In [19]:
# tmpl=288
# outl=12
# file_name = f'index_in{tmpl}_out{outl}.pkl'
# with open(file_name, 'rb') as file:
#     data = pickle.load(file)
# print(len(data['valid']))